# Chapter 3.2 EB 20D Main Flow Matching

This tutorial moves from the toy mechanism to the main EB timecourse experiment. We load EB snapshots, audit the selected 20D representation, split source and target cells, sample independent endpoint pairs, train the 20D `VelocityMLP`, cache the model for Chapter 3.3, and inspect Euler sampling sensitivity.


## Tutorial setup

The setup cells keep reusable infrastructure here: project-root discovery, deterministic seeds, output directories, and save/display helpers. The scientific objects are introduced in the experiment sections below.


In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

try:
    import torch
except ImportError as exc:
    raise ImportError("This notebook requires PyTorch for CFM training and sampling.") from exc

CWD = Path.cwd().resolve()
root_candidates = [
    CWD,
    CWD.parent,
    CWD.parent.parent,
    CWD / "flow_matching_for_dynamic_biology",
    CWD.parent / "flow_matching_for_dynamic_biology",
    CWD.parent.parent / "flow_matching_for_dynamic_biology",
]
for candidate in root_candidates:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError("Could not locate project root containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ch03_tutorial as ch03
from src.utils import set_seed
from src.data import load_eb_timecourse_for_ch03, copy_trajectorynet_eb_to_data
from src.models import VelocityMLP, count_parameters
from src.losses import cfm_batch, cfm_loss_from_pairs
from src.train import train_cfm_steps
from src.sampling import euler_sample, midpoint_sample, odeint_sample
from src.metrics import endpoint_metrics, mmd_rbf, sliced_wasserstein_distance, trajectory_straightness
from src.plots import plot_nfe_vs_error


In [ ]:
SEED = int(os.environ.get("CH03_SEED", "42"))
set_seed(SEED)
rng = np.random.default_rng(SEED)

QUICK_MODE = os.environ.get("CH03_QUICK", "1") == "1"
SMOKE_MODE = os.environ.get("CH03_SMOKE_MODE", "0") == "1"
PAPER_FIGURE_MODE = os.environ.get("CH03_PAPER_FIGURE_MODE", "1") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FULL_RUN_HINTS = {
    "eb_train_steps": 10000,
    "cnf_steps": 600,
    "time_ablation_steps": 1200,
    "capacity_steps": 1200,
    "notes": "Set CH03_QUICK=0 for full-run settings; default remains quick and reproducible.",
}

context = ch03.make_ch03_context(PROJECT_ROOT)
FIG_DIR = context.fig_dir
TABLE_DIR = context.table_dir
OUT_DIR = context.output_dir
PAPER_COLORS = ch03.PAPER_COLORS

figures_written = []
paper_ready_png_written = []
paper_ready_pdf_written = []
tables_written = []
paper_tables_written = []
skipped_items = []

set_paper_style = ch03.set_paper_style
add_panel_label = ch03.add_panel_label
short_strategy_label = ch03.short_strategy_label
clean_spines = ch03.clean_spines
format_metric_axis = ch03.format_metric_axis
add_note = ch03.add_note
as_np = ch03.as_np
subsample_idx = ch03.subsample_idx
robust_limits = ch03.robust_limits
format_axis = ch03.format_axis

set_paper_style()
print({"project_root": str(PROJECT_ROOT), "device": DEVICE, "quick_mode": QUICK_MODE, "smoke_mode": SMOKE_MODE, "seed": SEED})


In [ ]:
from src.artifacts import safe_relpath


def _rel(path):
    return safe_relpath(path, root=PROJECT_ROOT)


def save_figure(fig, filename, dpi=300, write_pdf=None):
    if write_pdf is None:
        write_pdf = PAPER_FIGURE_MODE
    png_path = ch03.save_figure(fig, FIG_DIR, filename, dpi=dpi, write_pdf=write_pdf)
    rel_png = _rel(png_path)
    if rel_png not in figures_written:
        figures_written.append(rel_png)
    if write_pdf:
        pdf_path = png_path.with_suffix(".pdf")
        rel_pdf = _rel(pdf_path)
        if rel_pdf not in paper_ready_pdf_written:
            paper_ready_pdf_written.append(rel_pdf)
    plt.close(fig)
    return png_path


def save_paper_figure(fig, stem, dpi=300):
    png_path = ch03.save_figure(fig, FIG_DIR, stem, dpi=dpi, write_pdf=True)
    pdf_path = png_path.with_suffix(".pdf")
    for path, bucket in [(png_path, paper_ready_png_written), (pdf_path, paper_ready_pdf_written)]:
        rel = _rel(path)
        if rel not in bucket:
            bucket.append(rel)
    rel_png = _rel(png_path)
    if rel_png not in figures_written:
        figures_written.append(rel_png)
    plt.close(fig)
    return png_path, pdf_path


def save_csv(frame, filename):
    path = ch03.save_csv(TABLE_DIR / filename, pd.DataFrame(frame))
    rel = _rel(path)
    if rel not in tables_written:
        tables_written.append(rel)
    return path


def save_paper_table(frame, stem, index=False):
    paths = ch03.save_paper_table(TABLE_DIR / stem, pd.DataFrame(frame), index=index)
    for path in paths:
        rel = _rel(path)
        if rel not in paper_tables_written:
            paper_tables_written.append(rel)
    return paths


def save_run_json(payload, filename):
    return ch03.save_json(OUT_DIR / filename, payload)


def display_saved_figure(path, width=None):
    return ch03.display_saved_figure(path, width=width)


def display_saved_figures(paths, width=None):
    return ch03.display_saved_figures(paths, width=width)


def display_table(frame, columns=None, n=10):
    return ch03.display_table(pd.DataFrame(frame), columns=columns, n=n)


## 1. Data audit

The EB main experiment uses `load_eb_timecourse_for_ch03`, which returns 20D PC coordinates as `X_cost` and 2D PHATE coordinates as `X_plot`. The model state is `X_cost[:, :20]`; PHATE is display-only and does not enter training, sampling metrics, MMD, or Sliced W2.


In [ ]:
EB_PATH = PROJECT_ROOT / "data" / "trajectorynet_eb" / "eb_velocity_v5.npz"
EB_SOURCE_PATH = PROJECT_ROOT.parent / "baselines" / "trajectorynet" / "data" / "eb_velocity_v5.npz"
if not EB_PATH.exists():
    print(f"{EB_PATH} not found; attempting copy from {EB_SOURCE_PATH}")
    copy_trajectorynet_eb_to_data(source_path=EB_SOURCE_PATH, target_path=EB_PATH)

max_cells_per_time = 750 if QUICK_MODE else 1800
if SMOKE_MODE:
    max_cells_per_time = 90

eb = load_eb_timecourse_for_ch03(
    EB_PATH,
    cost_embedding="pcs",
    plot_embedding="phate",
    n_cost_dims=20,
    max_cells_per_time=max_cells_per_time,
    seed=SEED,
)

data_audit = pd.DataFrame([
    {"object": "X_cost", "shape": tuple(eb["X_cost"].shape), "role": "20D PC training and metric space"},
    {"object": "X_plot", "shape": tuple(eb["X_plot"].shape), "role": "2D PHATE display only"},
    {"object": "time", "shape": tuple(eb["time"].shape), "role": "snapshot labels for adjacent source/target selection"},
])
display_table(data_audit, n=3)


In [ ]:
selected_counts = pd.Series(eb["time"], name="time").value_counts().sort_index().rename_axis("time").reset_index(name="selected_n_cells")
count_table = eb["full_counts_by_time"].merge(selected_counts, on="time", how="left")
count_table["selected_n_cells"] = count_table["selected_n_cells"].fillna(0).astype(int)
save_csv(count_table, "ch03_eb_timepoint_counts.csv")
display_table(count_table[["time", "n_cells", "selected_n_cells"]], n=8)


In [ ]:
time_labels = sorted(pd.Series(eb["time"]).unique(), key=lambda x: int(x) if str(x).isdigit() else str(x))
if len(time_labels) < 2:
    raise ValueError(f"Need at least two EB time labels; found {time_labels}")
source_time = str(time_labels[0])
target_time = str(time_labels[1])

mask_source = eb["time"].astype(str) == source_time
mask_target = eb["time"].astype(str) == target_time
X_source_20d = np.asarray(eb["X_cost"][mask_source], dtype=np.float32)
X_target_20d = np.asarray(eb["X_cost"][mask_target], dtype=np.float32)
X_source_phate = np.asarray(eb["X_plot"][mask_source], dtype=np.float32)
X_target_phate = np.asarray(eb["X_plot"][mask_target], dtype=np.float32)
cell_source = np.asarray(eb["cell_id"])[mask_source]
cell_target = np.asarray(eb["cell_id"])[mask_target]

assert X_source_20d.shape[1] == 20 and X_target_20d.shape[1] == 20
assert X_source_phate.shape[1] == 2 and X_target_phate.shape[1] == 2
endpoint_audit = pd.DataFrame([
    {"population": "source", "time": source_time, "X_20d_shape": tuple(X_source_20d.shape), "X_phate_shape": tuple(X_source_phate.shape)},
    {"population": "target", "time": target_time, "X_20d_shape": tuple(X_target_20d.shape), "X_phate_shape": tuple(X_target_phate.shape)},
])
display_table(endpoint_audit, n=2)


## 2. Train/validation split

The split is by cells, not by generated endpoint pairs. Source cells are split 80/20, target cells are split 80/20, training pairs are sampled only from train source and train target cells, and validation pairs are sampled only from held-out source and target cells.


In [ ]:
def train_val_indices(n, train_fraction=0.8, seed=42):
    local_rng = np.random.default_rng(seed)
    perm = local_rng.permutation(n)
    n_train = int(np.floor(float(train_fraction) * n))
    n_train = min(max(n_train, 1), n - 1)
    train = np.sort(perm[:n_train])
    val = np.sort(perm[n_train:])
    return train, val

src_train_idx, src_val_idx = train_val_indices(len(X_source_20d), seed=SEED + 10)
tgt_train_idx, tgt_val_idx = train_val_indices(len(X_target_20d), seed=SEED + 11)

X0_train = X_source_20d[src_train_idx]
X1_train = X_target_20d[tgt_train_idx]
X0_val = X_source_20d[src_val_idx]
X1_val = X_target_20d[tgt_val_idx]
X0_train_phate = X_source_phate[src_train_idx]
X1_train_phate = X_target_phate[tgt_train_idx]
X0_val_phate = X_source_phate[src_val_idx]
X1_val_phate = X_target_phate[tgt_val_idx]

split_table = pd.DataFrame([
    {"population": "source", "time": source_time, "train_cells": len(X0_train), "val_cells": len(X0_val)},
    {"population": "target", "time": target_time, "train_cells": len(X1_train), "val_cells": len(X1_val)},
])
save_csv(split_table, "ch03_eb20d_train_val_split.csv")
display_table(split_table, n=4)


## 3. Endpoint pairing

CFM supervision uses independent source and target draws. The validation MSE below uses a fixed held-out endpoint-pair set and a fixed interpolation-time grid, so validation curves are deterministic across logging events.


In [ ]:
pair_rng = np.random.default_rng(SEED + 12)


def pair_batch_fn(batch_size):
    idx0 = pair_rng.integers(0, len(X0_train), size=int(batch_size))
    idx1 = pair_rng.integers(0, len(X1_train), size=int(batch_size))
    return {"x0": X0_train[idx0].astype(np.float32), "x1": X1_train[idx1].astype(np.float32)}


val_pair_n = 1200 if QUICK_MODE else 2500
if SMOKE_MODE:
    val_pair_n = 180
val_pair_rng = np.random.default_rng(SEED + 13)
val_i0 = val_pair_rng.integers(0, len(X0_val), size=int(val_pair_n))
val_i1 = val_pair_rng.integers(0, len(X1_val), size=int(val_pair_n))
val_x0 = X0_val[val_i0].astype(np.float32)
val_x1 = X1_val[val_i1].astype(np.float32)
val_t_grid = np.asarray([0.25, 0.50, 0.75], dtype=np.float32)

pairing_table = pd.DataFrame([
    {"pair_set": "train", "source_pool": len(X0_train), "target_pool": len(X1_train), "sampling": "fresh independent draws each batch"},
    {"pair_set": "validation", "source_pool": len(X0_val), "target_pool": len(X1_val), "sampling": f"fixed {val_pair_n} held-out pairs"},
])
display_table(pairing_table, n=2)


## 4. Model config

The EB model is trained in 20D PC space using `VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4)`. The loss is local CFM velocity regression in 20D; no PHATE coordinates appear in the model or loss.


In [ ]:
set_seed(SEED + 20)
eb_model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
eb_optimizer = torch.optim.Adam(eb_model.parameters(), lr=1e-3)

eb_steps = 1500 if QUICK_MODE else 10000
if SMOKE_MODE:
    eb_steps = 45
eb_batch_size = 256
if SMOKE_MODE:
    eb_batch_size = 96
log_every = 100 if not SMOKE_MODE else 10

model_config_table = pd.DataFrame([
    {"parameter": "x_dim", "value": 20},
    {"parameter": "hidden_dim", "value": 256},
    {"parameter": "hidden_layers", "value": 4},
    {"parameter": "train_steps", "value": eb_steps},
    {"parameter": "batch_size", "value": eb_batch_size},
    {"parameter": "log_every", "value": log_every},
    {"parameter": "n_parameters", "value": count_parameters(eb_model)},
    {"parameter": "device", "value": DEVICE},
])
display_table(model_config_table, n=8)


## 5. Training loop

This is the core EB CFM experiment. The loop is visible because the optimization objective, held-out validation call, logging cadence, and wall-clock accounting are part of the scientific method.


In [ ]:
def val_cfm_mse(model, x0_np, x1_np, t_values, device="cpu", max_eval_pairs=None):
    model.eval()
    if max_eval_pairs is not None and len(x0_np) > max_eval_pairs:
        idx = subsample_idx(len(x0_np), max_eval_pairs, seed=SEED + 14)
        x0_np = x0_np[idx]
        x1_np = x1_np[idx]
    x0_t = torch.as_tensor(x0_np, dtype=torch.float32, device=device)
    x1_t = torch.as_tensor(x1_np, dtype=torch.float32, device=device)
    losses = []
    with torch.no_grad():
        for t_value in t_values:
            t_t = torch.full((x0_t.shape[0], 1), float(t_value), dtype=torch.float32, device=device)
            loss = cfm_loss_from_pairs(model, x0_t, x1_t, s=t_t)
            losses.append(float(loss.detach().cpu()))
    model.train()
    return float(np.mean(losses))


In [ ]:
rows = []
eb_model.train()
start = time.perf_counter()
last_log_step = 0
last_log_time = start
for step in range(1, eb_steps + 1):
    batch = pair_batch_fn(eb_batch_size)
    x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
    x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
    loss = cfm_loss_from_pairs(eb_model, x0, x1)
    eb_optimizer.zero_grad()
    loss.backward()
    eb_optimizer.step()

    if step == 1 or step % log_every == 0 or step == eb_steps:
        now = time.perf_counter()
        val_mse = val_cfm_mse(eb_model, val_x0, val_x1, val_t_grid, device=DEVICE, max_eval_pairs=1200 if not SMOKE_MODE else 180)
        rows.append({
            "step": int(step),
            "train_loss_20d": float(loss.detach().cpu()),
            "val_mse_20d_fixed_t_grid": float(val_mse),
            "wall_time_sec": float(now - start),
            "sec_per_step_since_last_log": float((now - last_log_time) / max(1, step - last_log_step)),
            "batch_size": int(eb_batch_size),
            "x_dim": 20,
            "hidden_dim": 256,
            "hidden_layers": 4,
            "n_parameters": count_parameters(eb_model),
            "device": DEVICE,
        })
        last_log_step = step
        last_log_time = now

eb_history = pd.DataFrame(rows)
save_csv(eb_history, "ch03_eb20d_training_log.csv")
print({"logged_rows": len(eb_history), "final_step": int(eb_history["step"].iloc[-1])})


## 6. Loss table

The training log is saved as the EB main loss table. Previewing the final rows makes the optimization scale and validation trend visible before plotting.


In [ ]:
loss_preview_columns = [
    "step",
    "train_loss_20d",
    "val_mse_20d_fixed_t_grid",
    "wall_time_sec",
    "sec_per_step_since_last_log",
]
display_table(eb_history.tail(6), columns=loss_preview_columns, n=6)


## 7. Loss figure

Figure B1 is the saved train/validation CFM loss curve. It is displayed immediately after saving so a fresh executed notebook contains the figure inline.


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.plot(eb_history["step"], eb_history["train_loss_20d"], marker="o", markersize=2.5, linewidth=1.4, color="#2F6B5E", label="train CFM MSE")
ax.plot(eb_history["step"], eb_history["val_mse_20d_fixed_t_grid"], marker="s", markersize=2.5, linewidth=1.4, color="#B9352B", label="val CFM MSE")
ax.set_xlabel("optimization step")
ax.set_ylabel("20D velocity-regression MSE")
ax.set_title("EB 20D CFM training and validation loss")
ax.grid(axis="y", color="0.92", linewidth=0.7)
ax.legend(frameon=False)
loss_figure_path = save_figure(fig, "figB1_eb20d_train_val_loss.png")
display_saved_figure(loss_figure_path, width=650)


## 8. Model cache

Chapter 3.3 reads the exact model and config names below. The default seed remains 42, so the default cache filenames stay `ch03_eb20d_velocity_mlp_seed42.pt` and `ch03_eb20d_main_config_seed42.json`.


In [ ]:
DEFAULT_MAIN_MODEL_ARTIFACT = "ch03_eb20d_velocity_mlp_seed42.pt"
DEFAULT_MAIN_CONFIG_ARTIFACT = "ch03_eb20d_main_config_seed42.json"
MAIN_MODEL_PATH = OUT_DIR / f"ch03_eb20d_velocity_mlp_seed{SEED}.pt"
MAIN_CONFIG_PATH = OUT_DIR / f"ch03_eb20d_main_config_seed{SEED}.json"
torch.save(eb_model.state_dict(), MAIN_MODEL_PATH)
main_model_config = {
    "seed": SEED,
    "quick_mode": QUICK_MODE,
    "smoke_mode": SMOKE_MODE,
    "x_dim": 20,
    "hidden_dim": 256,
    "hidden_layers": 4,
    "steps": int(eb_steps),
    "batch_size": int(eb_batch_size),
    "source_time": source_time,
    "target_time": target_time,
    "model_path": str(MAIN_MODEL_PATH.relative_to(PROJECT_ROOT)),
}
save_run_json(main_model_config, MAIN_CONFIG_PATH.name)
model_cache_table = pd.DataFrame([
    {"artifact": "main model state_dict", "path": str(MAIN_MODEL_PATH.relative_to(PROJECT_ROOT))},
    {"artifact": "main model config", "path": str(MAIN_CONFIG_PATH.relative_to(PROJECT_ROOT))},
])
display_table(model_cache_table, n=2)


## 9. Endpoint pair visualization

Figure 3.4 draws sampled train endpoint pairs in PHATE 2D for interpretability. The CFM training paths are straight lines in 20D PC space; their PHATE-projected line segments are visualization-only chords.


In [ ]:
def plot_endpoint_pairs_phate(X0_plot, X1_plot, n_pairs=80, seed=42):
    local_rng = np.random.default_rng(seed)
    n_pairs = min(int(n_pairs), max(len(X0_plot), len(X1_plot)))
    idx0 = local_rng.integers(0, len(X0_plot), size=n_pairs)
    idx1 = local_rng.integers(0, len(X1_plot), size=n_pairs)
    x0p = X0_plot[idx0]
    x1p = X1_plot[idx1]
    xlim, ylim = robust_limits(X0_plot, X1_plot, x0p, x1p, margin=0.10)
    fig, ax = plt.subplots(figsize=(5.6, 4.6))
    sidx = subsample_idx(len(X0_plot), 650, seed=91)
    tidx = subsample_idx(len(X1_plot), 650, seed=92)
    ax.scatter(X0_plot[sidx, 0], X0_plot[sidx, 1], s=7, color="#3267A8", alpha=0.38, linewidths=0, label=f"source time {source_time}")
    ax.scatter(X1_plot[tidx, 0], X1_plot[tidx, 1], s=7, color="#C9463D", alpha=0.34, linewidths=0, label=f"target time {target_time}")
    for a, b in zip(x0p, x1p):
        ax.plot([a[0], b[0]], [a[1], b[1]], color="0.25", alpha=0.24, linewidth=0.55)
    format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title="Sampled EB train endpoint pairs displayed in PHATE")
    ax.legend(frameon=False, loc="best")
    add_note(ax, "20D training chords; PHATE display only", loc="lower left")
    return fig

pair_plot_n = 80 if QUICK_MODE else 120
if SMOKE_MODE:
    pair_plot_n = 35
fig = plot_endpoint_pairs_phate(X0_train_phate, X1_train_phate, n_pairs=pair_plot_n, seed=SEED + 30)
pair_figure_path = save_figure(fig, "fig03_04_eb_endpoint_pairs_phate.png")
display_saved_figure(pair_figure_path, width=650)


## 10. Sampling and population evolution

Sampling integrates the learned velocity field in 20D PC space with `src.sampling.euler_sample`. Generated 20D states are projected to PHATE-like coordinates only for visualization.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_neighbors = min(20, len(eb["X_cost"]))
phate_projector = KNeighborsRegressor(n_neighbors=knn_neighbors, weights="distance")
phate_projector.fit(np.asarray(eb["X_cost"], dtype=np.float32), np.asarray(eb["X_plot"], dtype=np.float32))

sample_n = min(1200 if QUICK_MODE else 2200, len(X0_val))
if SMOKE_MODE:
    sample_n = min(150, len(X0_val))
sample_idx = subsample_idx(len(X0_val), sample_n, seed=SEED + 31)
x0_eval_20d = torch.as_tensor(X0_val[sample_idx], dtype=torch.float32, device=DEVICE)
sampling_plan = pd.DataFrame([
    {"item": "sample_n", "value": sample_n},
    {"item": "projector", "value": f"kNN X_cost->X_plot, k={knn_neighbors}"},
    {"item": "sampling_space", "value": "20D PC"},
])
display_table(sampling_plan, n=3)


In [ ]:
eb_model.eval()
euler_population_steps = 100 if not SMOKE_MODE else 25
with torch.no_grad():
    eb_final_20d_t, eb_traj_20d_t, eb_nfe = euler_sample(eb_model, x0_eval_20d, n_steps=euler_population_steps, return_traj=True)
eb_traj_20d = as_np(eb_traj_20d_t)
eb_final_20d = as_np(eb_final_20d_t)

show_times = [0.0, 0.25, 0.5, 0.75, 1.0]
show_steps = np.round(np.asarray(show_times) * (eb_traj_20d.shape[0] - 1)).astype(int)
generated_phate_by_time = [phate_projector.predict(eb_traj_20d[step]) for step in show_steps]

xlim, ylim = robust_limits(X_source_phate, X_target_phate, *generated_phate_by_time, margin=0.10)
fig, axes = plt.subplots(1, 5, figsize=(12.8, 2.85), sharex=True, sharey=True)
for ax, tau, gen_plot in zip(axes, show_times, generated_phate_by_time):
    if tau == 1.0:
        tidx = subsample_idx(len(X_target_phate), 800, seed=101)
        ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=5, color="0.65", alpha=0.20, linewidths=0, label="real target")
    ax.scatter(gen_plot[:, 0], gen_plot[:, 1], s=6, color="#2F6B5E", alpha=0.50, linewidths=0, label="generated")
    format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title=f"t={tau:.2f}")
    if ax is not axes[0]:
        ax.set_ylabel("")
add_note(axes[0], "20D Euler states\nPHATE display only", loc="lower left")
population_figure_path = save_figure(fig, "fig03_08_eb_population_evolution_phate.png")
display_saved_figure(population_figure_path, width=900)
print({"euler_nfe": eb_nfe, "trajectory_shape": eb_traj_20d.shape})


## 11. Euler step sensitivity

This diagnostic fixes the trained 20D EB model and varies the Euler step count. CSV metrics are computed in 20D: MMD to a fine-step reference, endpoint MMD to held-out target cells, and Sliced W2 to held-out target cells.


In [ ]:
euler_Ks = [1, 2, 5, 10, 20, 50, 100, 200]
reference_K = 200 if QUICK_MODE else 500
if SMOKE_MODE:
    euler_Ks = [1, 2, 5, 10, 20]
    reference_K = 20

with torch.no_grad():
    ref_endpoint_t, _, ref_nfe = euler_sample(eb_model, x0_eval_20d, n_steps=reference_K, return_traj=False)
ref_endpoint_20d = as_np(ref_endpoint_t)

target_eval_20d = X1_val.astype(np.float32)
euler_rows = []
euler_panels = []
for K in euler_Ks:
    tic = time.perf_counter()
    with torch.no_grad():
        endpoint_t, _, nfe = euler_sample(eb_model, x0_eval_20d, n_steps=K, return_traj=False)
    wall = time.perf_counter() - tic
    endpoint_20d = as_np(endpoint_t)
    projected = phate_projector.predict(endpoint_20d)
    euler_panels.append({"K": K, "nfe": nfe, "endpoint_20d": endpoint_20d, "endpoint_phate": projected})
    euler_rows.append({
        "solver": "euler",
        "K": int(K),
        "nfe": int(nfe),
        "reference_K": int(reference_K),
        "wall_time_sec": float(wall),
        "mmd_20d_to_reference": float(mmd_rbf(endpoint_20d, ref_endpoint_20d)),
        "endpoint_mmd_20d_to_target_val": float(mmd_rbf(endpoint_20d, target_eval_20d)),
        "sliced_w2_20d_to_target_val": float(sliced_wasserstein_distance(endpoint_20d, target_eval_20d, n_projections=128 if not SMOKE_MODE else 48, seed=SEED + K)),
    })

euler_table = pd.DataFrame(euler_rows)
save_csv(euler_table, "ch03_euler_step_sensitivity.csv")
display_table(euler_table, n=10)


In [ ]:
ncols = 4
nrows = int(np.ceil(len(euler_panels) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 3.0 * nrows), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(-1)
xlim, ylim = robust_limits(X_target_phate, *(p["endpoint_phate"] for p in euler_panels), margin=0.10)
for ax, panel in zip(axes, euler_panels):
    tidx = subsample_idx(len(X_target_phate), 650, seed=111)
    ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=5, color="0.68", alpha=0.18, linewidths=0)
    pts = panel["endpoint_phate"]
    ax.scatter(pts[:, 0], pts[:, 1], s=6, color="#2F6B5E", alpha=0.50, linewidths=0)
    format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title=f"Euler K={panel['K']}\nNFE={panel['nfe']}")
for ax in axes[len(euler_panels):]:
    ax.axis("off")
fig.suptitle("Euler step sensitivity: 20D endpoints projected to PHATE for display")
euler_figure_path = save_figure(fig, "fig03_09_euler_step_sensitivity_phate.png")
display_saved_figure(euler_figure_path, width=900)


## Artifact audit

This final cell records the EB main-flow artifact contract. It checks every owned figure, table, and cache/output path, saves the split manifest, then checks that the manifest itself is present and non-empty.


In [ ]:
expected_figures = [
    FIG_DIR / "figB1_eb20d_train_val_loss.png",
    FIG_DIR / "fig03_04_eb_endpoint_pairs_phate.png",
    FIG_DIR / "fig03_08_eb_population_evolution_phate.png",
    FIG_DIR / "fig03_09_euler_step_sensitivity_phate.png",
]
expected_tables = [
    TABLE_DIR / "ch03_eb_timepoint_counts.csv",
    TABLE_DIR / "ch03_eb20d_train_val_split.csv",
    TABLE_DIR / "ch03_eb20d_training_log.csv",
    TABLE_DIR / "ch03_euler_step_sensitivity.csv",
]
run_config_path = OUT_DIR / "run_config_03_2_eb20d_main_flow_matching.json"
manifest_path = OUT_DIR / "artifact_manifest_03_2_eb20d_main_flow_matching.csv"
expected_outputs = [
    OUT_DIR / f"ch03_eb20d_velocity_mlp_seed{SEED}.pt",
    OUT_DIR / f"ch03_eb20d_main_config_seed{SEED}.json",
    run_config_path,
]


def raise_for_missing_artifacts(manifest):
    missing = manifest.loc[~manifest["exists"] | manifest["bytes"].eq(0)]
    if not missing.empty:
        raise FileNotFoundError("Missing or empty EB main artifacts:\n" + missing.to_string(index=False))

save_run_json(
    {
        "seed": SEED,
        "quick_mode": QUICK_MODE,
        "smoke_mode": SMOKE_MODE,
        "owned_figures": [p.name for p in expected_figures],
        "owned_tables": [p.name for p in expected_tables],
        "owned_outputs": [p.name for p in expected_outputs],
    },
    run_config_path.name,
)
artifact_manifest = ch03.check_required_artifacts(
    expected_figures=expected_figures,
    expected_tables=expected_tables,
    expected_outputs=expected_outputs,
)
ch03.save_csv(manifest_path, artifact_manifest)
expected_outputs.append(manifest_path)
artifact_manifest = ch03.check_required_artifacts(
    expected_figures=expected_figures,
    expected_tables=expected_tables,
    expected_outputs=expected_outputs,
)
raise_for_missing_artifacts(artifact_manifest)
display(artifact_manifest)
